In [12]:
import os
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scistree2 as s2


print("Software versions used for this notebook run:")
print(f"Python:    {sys.version.split()[0]}")
print(f"NumPy:     {np.__version__}")
print(f"pandas:    {pd.__version__}")
print(f"ScisTree2 Python package: {version('scistree2')}")


def compute_noise_rates(D, E):
    """
    Estimate genotype error rates from observed genotype matrix D and 
    inferred conflict-free (i.e., perfect phylogeny) matrix E.

    Both DataFrames must have cells as rows and mutations as columns,
    with identical labels and ordering. Genotypes are encoded as
    0 (absent), 1 (present), and 3 (missing).

    The false-positive rate is the fraction of observed presences among
    entries inferred to be absent. The false-negative rate is the fraction
    of observed absences among entries inferred to be present.

    Only pairs with both genotypes in {0, 1} contribute to these estimates.
    A rate is set to 0.0 when its denominator is zero.

    Returns (est_alpha, est_beta).
    """
    assert D.index.equals(E.index), "Cell IDs do not match"
    assert D.columns.equals(E.columns), "Mutation IDs do not match"

    # Count agreements and discrepancies, excluding missing entries.
    false_positives = ((D == 1) & (E == 0)).sum().sum()
    true_negatives = ((D == 0) & (E == 0)).sum().sum()

    false_negatives = ((D == 0) & (E == 1)).sum().sum()
    true_positives = ((D == 1) & (E == 1)).sum().sum()

    est_alpha = 0.0
    if false_positives + true_negatives > 0:
        est_alpha = false_positives / (false_positives + true_negatives)

    est_beta = 0.0
    if false_negatives + true_positives > 0:
        est_beta = false_negatives / (false_negatives + true_positives)

    return est_alpha, est_beta


def calculate_log_likelihood(D, E, alpha, beta, include_NA=True):
    """
    Calculate the natural-log likelihood of observed D given inferred E.

    Both DataFrames must have cells as rows and mutations as columns,
    with identical labels and ordering. The parameters alpha and beta
    are the false-positive and false-negative genotype error rates.

    Pairs with both genotypes in {0, 1} contribute transition terms.
    If include_NA is True, each missing entry in D (coded as 3) contributes
    log(0.5), regardless of its inferred genotype. Otherwise, missing
    entries contribute nothing.

    Returns negative infinity if an observed transition has probability
    zero. Higher scores indicate greater likelihood.
    """
    assert D.index.equals(E.index), "Cell IDs do not match"
    assert D.columns.equals(E.columns), "Mutation IDs do not match"

    if not (0 <= alpha <= 1 and 0 <= beta <= 1):
        raise ValueError("alpha and beta must be between 0 and 1.")

    score = 0.0

    # Each tuple contains: observed genotype, inferred genotype, probability.
    transitions = [
        (0, 1, beta),
        (1, 1, 1 - beta),
        (1, 0, alpha),
        (0, 0, 1 - alpha),
    ]

    for observed, inferred, probability in transitions:
        count = ((D == observed) & (E == inferred)).sum().sum()

        if count == 0:
            continue

        # An observed event with zero probability gives zero likelihood.
        if probability == 0:
            return float("-inf")

        score += count * np.log(probability)

    # Add the fixed contribution for missing observed genotypes.
    if include_NA:
        missing_count = (D == 3).sum().sum()
        score += missing_count * np.log(0.5)

    return float(score)

Software versions used for this notebook run:
Python:    3.10.16
NumPy:     1.26.4
pandas:    2.2.3
ScisTree2 Python package: 0.2.0


In [2]:
# Load alternate-read counts and total read depths.
# In the input files, rows are mutation sites and columns are cells.

variant_counts_file = "ad_alt_per_cell_nist_variants.tsv"
total_counts_file = "dp_per_cell_nist_variants.tsv"

variant_counts = pd.read_csv(variant_counts_file, sep="\t", index_col="Mut")
total_counts = pd.read_csv(total_counts_file, sep="\t", index_col="Mut")

# Transpose so that rows correspond to cells and columns to mutation sites.
total_counts = total_counts.T
variant_counts = variant_counts.T

# Standardize mutation identifiers by removing the leading "chr" and
# replacing underscores with hyphens.
# Example: "chr1_1886301_G_A" becomes "1-1886301-G-A".
total_counts.columns = (
    total_counts.columns
    .str.replace(r"^chr", "", regex=True)
    .str.replace("_", "-")
)
variant_counts.columns = (
    variant_counts.columns
    .str.replace(r"^chr", "", regex=True)
    .str.replace("_", "-")
)

# Shorten cell identifiers by removing the shared prefix.
# Example: "NIST-PL01-SC19" becomes "C19".
prefix = "NIST-PL01-S"

assert total_counts.index.str.startswith(prefix).all(), (
    "Not all entries have the required prefix!"
)
assert variant_counts.index.str.startswith(prefix).all(), (
    "Not all entries have the required prefix!"
)

total_counts.index = total_counts.index.str.replace(prefix, "", regex=False)
variant_counts.index = variant_counts.index.str.replace(prefix, "", regex=False)

In [3]:
cell_ids = total_counts.index
mut_ids = total_counts.columns

# Construct the observed genotype matrix: rows are cells, columns are mutations.
# Genotype codes: 0 = absent, 1 = present, 3 = missing.
D = pd.DataFrame(index=cell_ids, columns=mut_ids)

min_variant_reads_to_call_mutation = 3

for cell_id in cell_ids:
    for mut_id in mut_ids:
        num_variant_reads = variant_counts.at[cell_id, mut_id]
        num_total_reads = total_counts.at[cell_id, mut_id]

        assert num_variant_reads <= num_total_reads, (
            "ERROR. Variant > Total counts."
        )

        vaf = None

        # Mark entries with fewer than four total reads as missing.
        if num_total_reads in [0, 1, 2, 3]:
            D.at[cell_id, mut_id] = 3
            continue

        # Calculate the variant allele fraction for sufficiently covered sites.
        vaf = (1.0 * num_variant_reads) / num_total_reads

        # Call a mutation present if supported by at least three alternate
        # reads and a variant allele fraction of at least 0.1; otherwise absent.
        if (
            num_variant_reads >= min_variant_reads_to_call_mutation
            and vaf >= 0.1
        ):
            D.at[cell_id, mut_id] = 1
        else:
            D.at[cell_id, mut_id] = 0

In [4]:
# Run ScisTree2 using all cells to identify cells with excessive
# putative false-positive counts, which may indicate unreliable mutation
# calls and affect the accuracy of reference phylogeny reconstruction.

# Encode each cell–mutation pair as "reference_count|alternate_count".
# These are raw read counts used as the input to ScisTree2.

all_read_counts_matrix = pd.DataFrame(index=cell_ids, columns=mut_ids)

for cell_id in cell_ids:
    for mut_id in mut_ids:
        ref_count = (
            total_counts.at[cell_id, mut_id]
            - variant_counts.at[cell_id, mut_id]
        )
        alt_count = variant_counts.at[cell_id, mut_id]
        all_read_counts_matrix.at[cell_id, mut_id] = (
            f"{ref_count}|{alt_count}"
        )

# Write the count matrix with mutation sites as rows and cells as columns.
temporary_counts_file = "ScisTree2-temporary-file.csv"

all_read_counts_matrix_transposed = all_read_counts_matrix.T
all_read_counts_matrix_transposed.to_csv(
    temporary_counts_file,
    index=True,
    header=True,
)

# Parameter grid for the ScisTree2 read-count interface.
# alpha: sequencing-error parameter; beta: allelic-dropout parameter.
alpha_values = [0.01, 0.001, 10**(-5), 10**(-8), 10**(-80)]
beta_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# Store inferred genotype matrices by parameter combination.
# Each stored DataFrame has cells as rows and mutation sites as columns.
inferred_genotypes = {}

for alpha in alpha_values:
    inferred_genotypes[alpha] = {}

    for beta in beta_values:
        print(
            f"Running ScisTree2 version {version('scistree2')} "
            f"for alpha = {alpha}, beta = {beta}"
        )

        # Convert read counts to genotype probabilities.
        gp = s2.probability.from_csv(
            temporary_counts_file,
            source="read",
            ado=beta,
            seqerr=alpha,
        )

        # Infer the cell phylogeny and corresponding genotype matrix.
        caller_spr = s2.ScisTree2(threads=8)
        tree_spr, imputed_genotype_spr, likelihood_spr = caller_spr.infer(gp)

        inferred_genotypes[alpha][beta] = imputed_genotype_spr.T

Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.05
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.1
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.15
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.2
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.25
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.3
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.35
Running ScisTree2 version 0.2.0 for alpha = 0.01, beta = 0.4
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.05
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.1
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.15
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.2
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.25
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.3
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.35
Running ScisTree2 version 0.2.0 for alpha = 0.001, beta = 0.4
Running 

In [5]:
def print_false_positives_by_cell(df_input_D, df_output_E):
    """
    Print cells ranked by decreasing putative false-positive count.

    A putative false positive is an entry with observed genotype 1 in
    df_input_D and inferred genotype 0 in df_output_E. Both DataFrames
    must have cells as rows and mutations as columns, with identical
    labels and ordering.

    Output columns:
        1. Rank
        2. Cell ID
        3. Number of false positives in the cell
        4. Percentage of all false positives occurring in the cell
        5. Cumulative percentage of false positives up to that rank

    Cells with no false positives are omitted.
    """
    assert df_input_D.index.equals(df_output_E.index), "Row IDs do not match"
    assert df_input_D.columns.equals(df_output_E.columns), "Column IDs do not match"

    # Identify observed presences that are inferred to be absent.
    fp = (df_input_D == 1) & (df_output_E == 0)

    # Count false positives per cell and across all cells.
    fp_counts = fp.sum(axis=1)
    total_fp = fp_counts.sum()

    # Keep cells with at least one false positive and sort descending.
    fp_counts = fp_counts[fp_counts > 0].sort_values(ascending=False)

    print(
        "Columns: Rank | Cell ID | False-positive count | "
        "% of all false positives | Cumulative % of all false positives"
    )

    cumulative_percentage = 0.0

    for rank, (cell_id, count) in enumerate(fp_counts.items(), start=1):
        percentage = 100 * count / total_fp
        cumulative_percentage += percentage

        print(
            f"{rank}.\t{cell_id}\t{count}\t"
            f"{percentage:.2f}%\t{cumulative_percentage:.2f}%"
        )


print("False positives summary for alpha=0.01 and beta=0.05.")
print_false_positives_by_cell(D, inferred_genotypes[0.01][0.05])

print("\n---------------------------------------------------------------------------\n")

print("False positives summary for alpha=1e-80 and beta=0.35.")
print_false_positives_by_cell(D, inferred_genotypes[1e-80][0.35])

False positives summary for alpha=0.01 and beta=0.05.
Columns: Rank | Cell ID | False-positive count | % of all false positives | Cumulative % of all false positives
1.	C293	204	9.42%	9.42%
2.	C309	181	8.36%	17.77%
3.	C116	180	8.31%	26.08%
4.	C83	180	8.31%	34.40%
5.	C243	170	7.85%	42.24%
6.	C362	170	7.85%	50.09%
7.	C03	159	7.34%	57.43%
8.	C319	156	7.20%	64.64%
9.	C344	142	6.56%	71.19%
10.	C181	135	6.23%	77.42%
11.	C207	119	5.49%	82.92%
12.	C133	85	3.92%	86.84%
13.	C261	59	2.72%	89.57%
14.	C317	56	2.59%	92.15%
15.	C277	39	1.80%	93.95%
16.	C104	27	1.25%	95.20%
17.	C119	21	0.97%	96.17%
18.	C117	10	0.46%	96.63%
19.	C165	6	0.28%	96.91%
20.	C240	6	0.28%	97.18%
21.	C47	5	0.23%	97.41%
22.	C52	4	0.18%	97.60%
23.	C164	3	0.14%	97.74%
24.	C108	3	0.14%	97.88%
25.	C323	3	0.14%	98.01%
26.	C274	2	0.09%	98.11%
27.	C250	2	0.09%	98.20%
28.	C341	2	0.09%	98.29%
29.	C298	2	0.09%	98.38%
30.	C191	2	0.09%	98.48%
31.	C244	2	0.09%	98.57%
32.	C217	2	0.09%	98.66%
33.	C167	2	0.09%	98.75%
34.	C159	2	0.09%	98.85%
35.

In [6]:
# Cells with excessive putative false-positive counts, including when
# the ScisTree2 sequencing-error parameter alpha is as low as 1e-80.
cells_with_high_number_of_fps = [
    "C293", "C362", "C309", "C243", "C83", "C116", "C03", "C319",
    "C344", "C181", "C207", "C133", "C261", "C317", "C277", "C104",
    "C119",
]

# Check whether this same set of cells has the highest false-positive
# counts in every reconstruction, regardless of their order within the set.
expected_cells = set(cells_with_high_number_of_fps)
n_top = len(expected_cells)

num_checked = 0
num_matches = 0
tied_solutions = []

for alpha in alpha_values:
    for beta in beta_values:
        E = inferred_genotypes[alpha][beta]

        assert D.index.equals(E.index), "Row IDs do not match"
        assert D.columns.equals(E.columns), "Column IDs do not match"

        # Count putative false positives per cell: observed = 1, inferred = 0.
        fp_counts = ((D == 1) & (E == 0)).sum(axis=1)
        fp_counts = fp_counts.sort_values(ascending=False, kind="stable")

        # Compare the set of highest-ranking cells with the expected set.
        top_cells = set(fp_counts.head(n_top).index)
        matches = top_cells == expected_cells

        num_checked += 1
        if matches:
            num_matches += 1

        status = "MATCH" if matches else "MISMATCH"
        print(f"alpha={alpha:g}, beta={beta:.2f}: {status}")

        if not matches:
            print(
                f"  Expected cells missing from the top {n_top}:",
                sorted(expected_cells - top_cells),
            )
            print(
                f"  Other cells appearing in the top {n_top}:",
                sorted(top_cells - expected_cells),
            )

        # Flag ties across the cutoff: these make membership ambiguous.
        if len(fp_counts) > n_top:
            if fp_counts.iloc[n_top - 1] == fp_counts.iloc[n_top]:
                tied_solutions.append((alpha, beta))
                print(
                    f"  WARNING: ranks {n_top} and {n_top + 1} are tied at",
                    fp_counts.iloc[n_top - 1],
                    "false positives.",
                )

print(f"\nMatching top-{n_top} sets: {num_matches}/{num_checked}")

if num_matches == num_checked and not tied_solutions:
    print(
        f"The same {n_top} cells uniquely form the top {n_top} "
        "by false-positive count in every solution."
    )
elif tied_solutions:
    print("Some solutions have ties at the cutoff; review the warnings above.")

alpha=0.01, beta=0.05: MATCH
alpha=0.01, beta=0.10: MATCH
alpha=0.01, beta=0.15: MATCH
alpha=0.01, beta=0.20: MATCH
alpha=0.01, beta=0.25: MATCH
alpha=0.01, beta=0.30: MATCH
alpha=0.01, beta=0.35: MATCH
alpha=0.01, beta=0.40: MATCH
alpha=0.001, beta=0.05: MATCH
alpha=0.001, beta=0.10: MATCH
alpha=0.001, beta=0.15: MATCH
alpha=0.001, beta=0.20: MATCH
alpha=0.001, beta=0.25: MATCH
alpha=0.001, beta=0.30: MATCH
alpha=0.001, beta=0.35: MATCH
alpha=0.001, beta=0.40: MATCH
alpha=1e-05, beta=0.05: MATCH
alpha=1e-05, beta=0.10: MATCH
alpha=1e-05, beta=0.15: MATCH
alpha=1e-05, beta=0.20: MATCH
alpha=1e-05, beta=0.25: MATCH
alpha=1e-05, beta=0.30: MATCH
alpha=1e-05, beta=0.35: MATCH
alpha=1e-05, beta=0.40: MATCH
alpha=1e-08, beta=0.05: MATCH
alpha=1e-08, beta=0.10: MATCH
alpha=1e-08, beta=0.15: MATCH
alpha=1e-08, beta=0.20: MATCH
alpha=1e-08, beta=0.25: MATCH
alpha=1e-08, beta=0.30: MATCH
alpha=1e-08, beta=0.35: MATCH
alpha=1e-08, beta=0.40: MATCH
alpha=1e-80, beta=0.05: MATCH
alpha=1e-80, beta=

In [7]:
# Identify cells with at most three total reads at 15% or more of mutation sites.
# These low-coverage entries are encoded as 3 (missing) in D.
low_coverage_cells = []

for cell_id in D.index:
    low_coverage_count = (D.loc[cell_id] == 3).sum()

    # Calculate the percentage across all analyzed mutation sites in this cell.
    low_coverage_percentage = 100 * low_coverage_count / D.shape[1]

    if low_coverage_percentage >= 15:
        low_coverage_cells.append(cell_id)

print(
    "Cells with at most 3 total reads at 15% or more of mutation sites:",
    ", ".join(low_coverage_cells) or "None"
)

Cells with at most 3 total reads at 15% or more of mutation sites: C103


In [8]:
# Exclude cells flagged for excessive putative false positives or low coverage.
cells_to_remove = cells_with_high_number_of_fps + low_coverage_cells

# Apply the same cell exclusions to both read-count matrices and D.
total_counts = total_counts.drop(index=cells_to_remove)
variant_counts = variant_counts.drop(index=cells_to_remove)
D = D.drop(index=cells_to_remove)

# Update the identifiers for subsequent tree reconstruction.
cell_ids = total_counts.index
mut_ids = total_counts.columns

In [9]:
# Reconstruct the reference phylogeny using the cells retained after filtering.

# Encode the filtered read counts as "reference_count|alternate_count".
all_read_counts_matrix = pd.DataFrame(index=cell_ids, columns=mut_ids)

for cell_id in cell_ids:
    for mut_id in mut_ids:
        ref_count = (
            total_counts.at[cell_id, mut_id]
            - variant_counts.at[cell_id, mut_id]
        )
        alt_count = variant_counts.at[cell_id, mut_id]
        all_read_counts_matrix.at[cell_id, mut_id] = (
            f"{ref_count}|{alt_count}"
        )

# Overwrite the temporary input file with counts for the retained cells.
# Rows correspond to mutation sites and columns to cells.
all_read_counts_matrix_transposed = all_read_counts_matrix.T
all_read_counts_matrix_transposed.to_csv(
    temporary_counts_file,
    index=True,
    header=True,
)

# Track the candidate with the lowest negative log-likelihood (NLL).
best_NLL = 10**10
best_alpha = 0
best_beta = 0
best_conflict_free_matrix = None

print(f"Number of cells in tree reconstruction: {D.shape[0]}")
print(f"Number of mutations in tree reconstruction: {D.shape[1]}")
print()

for alpha in alpha_values:
    for beta in beta_values:
        print(f"Alpha = {alpha}")
        print(f"Beta  = {beta}")

        # Convert read counts to genotype probabilities using the current
        # sequencing-error (alpha) and allelic-dropout (beta) parameters.
        gp = s2.probability.from_csv(
            temporary_counts_file,
            source="read",
            ado=beta,
            seqerr=alpha,
        )

        # Infer a cell phylogeny and its corresponding genotype matrix.
        caller_spr = s2.ScisTree2(threads=8)
        tree_spr, imputed_genotype_spr, likelihood_spr = caller_spr.infer(gp)

        # Orient the inferred matrix to match D: cells as rows,
        # mutation sites as columns.
        conflict_free_matrix = imputed_genotype_spr.T

        # Estimate genotype error rates separately for each candidate.
        # Candidate selection uses the custom NLL evaluated at these rates.
        est_alpha, est_beta = compute_noise_rates(D, conflict_free_matrix)
        estimated_NLL = -calculate_log_likelihood(
            D,
            conflict_free_matrix,
            est_alpha,
            est_beta,
        )

        print(f"Negative log-likelihood (NLL) = {estimated_NLL}")

        # Retain the best genotype matrix and its generating grid parameters.
        # Strict comparison preserves the first candidate when scores tie.
        if estimated_NLL < best_NLL:
            best_NLL = estimated_NLL
            best_alpha, best_beta = alpha, beta
            best_conflict_free_matrix = conflict_free_matrix.copy(deep=True)

        print(
            "-----------------------------------------------------------------------------------\n"
        )

# Remove the temporary read-count input after completing the grid search.
os.remove(temporary_counts_file)

print(
    f"The best (lowest) NLL is attained for alpha={best_alpha} "
    f"and beta={best_beta}"
)

# Export the selected genotype matrix for subsequent analysis.
best_matrix_file = "best_NLL_solution_conflict_free_matrix.tsv"
best_conflict_free_matrix.to_csv(
    best_matrix_file,
    sep="\t",
    index=True,
    header=True,
)

print(
    "Conflict-free (perfect phylogeny) matrix corresponding to the "
    f"best NLL solution was written to {best_matrix_file}"
)

Number of cells in tree reconstruction: 101
Number of mutations in tree reconstruction: 7844

Alpha = 0.01
Beta  = 0.05
Negative log-likelihood (NLL) = 198043.795684884
-----------------------------------------------------------------------------------

Alpha = 0.01
Beta  = 0.1
Negative log-likelihood (NLL) = 198053.45310698394
-----------------------------------------------------------------------------------

Alpha = 0.01
Beta  = 0.15
Negative log-likelihood (NLL) = 198086.1952985456
-----------------------------------------------------------------------------------

Alpha = 0.01
Beta  = 0.2
Negative log-likelihood (NLL) = 198083.50117513174
-----------------------------------------------------------------------------------

Alpha = 0.01
Beta  = 0.25
Negative log-likelihood (NLL) = 198070.03024803975
-----------------------------------------------------------------------------------

Alpha = 0.01
Beta  = 0.3
Negative log-likelihood (NLL) = 198096.2208538858
--------------------------